# 🏥 CareFlow — Notebook 10: Markov Chain Pathway Modeling
## Discrete-Time Markov Chains (DTMC), Transition Matrices & Absorbing States

**Author:** CareFlow Process Intelligence Team  
**Dataset:** `careflow_analytics/output/stg_ehr_events.parquet`

---

### Objectives:
1. Estimate the 1-step empirical transition probability matrix $P_{ij} = \frac{C_{ij}}{\sum_k C_{ik}}$.
2. Decompose states into Transient states ($Q$) and Absorbing states ($R$).
3. Compute the **Fundamental Matrix** $N = (I - Q)^{-1}$ to find the expected number of visits per clinical state.
4. Calculate Absorption Probabilities $B = N R$ predicting final disposition (Discharge vs Admission).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

stg_df = pd.read_parquet('../careflow_analytics/output/stg_ehr_events.parquet')
trans = stg_df[stg_df["previous_activity"].notna()].copy()

activities = sorted(list(set(trans["previous_activity"].unique()).union(set(trans["activity_name"].unique()))))
print(f"Identified {len(activities)} clinical states: {activities}")


In [ ]:
# 1. Transition Frequency & Stochastic Probability Matrix
c_matrix = pd.crosstab(
    trans["previous_activity"],
    trans["activity_name"]
).reindex(index=activities, columns=activities, fill_value=0)

# Row-normalize to get stochastic probabilities
p_matrix = c_matrix.div(c_matrix.sum(axis=1).replace(0, 1), axis=0)

plt.figure(figsize=(10, 8))
sns.heatmap(p_matrix, annot=True, fmt='.2f', cmap='Blues', cbar=True)
plt.title("1-Step Discrete-Time Markov Transition Probability Matrix (P)")
plt.xlabel("Target Clinical State")
plt.ylabel("Source Clinical State")
plt.tight_layout()
plt.show()


In [ ]:
# 2. Absorbing Markov Chain Analysis
# Identify transient states vs absorbing states
absorbing_states = ['Discharge', 'Admission']
transient_states = [a for a in activities if a not in absorbing_states and a in p_matrix.index]

Q = p_matrix.loc[transient_states, transient_states].values
R = p_matrix.loc[transient_states, [s for s in absorbing_states if s in p_matrix.columns]].values

I = np.eye(len(transient_states))
# Fundamental matrix N = (I - Q)^(-1)
try:
    N = np.linalg.inv(I - Q)
    fundamental_df = pd.DataFrame(N, index=transient_states, columns=transient_states)
    print("Fundamental Matrix N (Expected visits before absorption):")
    print(fundamental_df.round(2))
    
    # Expected steps to absorption
    t_steps = N.sum(axis=1)
    steps_df = pd.DataFrame({'Expected Steps to Disposition': t_steps}, index=transient_states)
    print("\nExpected Trajectory Length by Starting State:")
    print(steps_df.round(2))
except np.linalg.LinAlgError:
    print("Matrix is singular; displaying empirical step lengths.")


In [ ]:
# 3. Absorption Probabilities B = N * R
try:
    B = N @ R
    b_df = pd.DataFrame(B, index=transient_states, columns=[s for s in absorbing_states if s in p_matrix.columns])
    print("Absorption Probabilities (Disposition Likelihood from State):")
    print(b_df.round(3))
except Exception as e:
    print(f"Absorption calculation: {e}")
